In [1]:
#load apikeys from .env

from dotenv import load_dotenv
import os
import getpass

load_dotenv()


if "ANTHROPIC_API_KEY" not in os.environ:
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Introduce tu  AWS Key: ")

In [2]:
#model agnostic function for llm inference with init_chat_model and invoke()

from langchain.chat_models import init_chat_model

def generate_example_orders(model_name, prompt, provider=None, token_limit=200, temperature = 0.7):
    model = init_chat_model(
        model_name,
        model_provider=provider,
        max_tokens=token_limit,
        temperature = temperature
    )
    response = model.invoke(prompt)
    return response

In [3]:
model_name = "claude-haiku-4-5-20251001"
provider = "anthropic"
prompt = "Generate the transcription of a pizza order"


response = generate_example_orders(
    model_name,
    prompt,
    provider
)

response

AIMessage(content="# Pizza Order Transcription\n\n**Customer:** Hi, I'd like to place an order for delivery.\n\n**Operator:** Sure! What can I get started for you today?\n\n**Customer:** I'd like a large pepperoni pizza and a medium vegetarian with extra mushrooms, please.\n\n**Operator:** Great! One large pepperoni and one medium veggie with extra mushrooms. Anything else?\n\n**Customer:** Yeah, can we add a 2-liter bottle of Coke and an order of garlic bread?\n\n**Operator:** Perfect. So that's a large pepperoni, medium vegetarian with extra mushrooms, one 2-liter Coke, and garlic bread. What's the delivery address?\n\n**Customer:** 247 Maple Street, Apartment 3B.\n\n**Operator:** 247 Maple Street, Apartment 3B. And what's", additional_kwargs={}, response_metadata={'id': 'msg_011Cf3JyqVi3traeVpQL7Jvi', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'max_tokens', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_toke

In [4]:
#Components of the AI Message

#response.content
#response.response_metadata
#response.tool_calls
#response.invalid_tool_calls
response.usage_metadata

{'input_tokens': 15,
 'output_tokens': 200,
 'total_tokens': 215,
 'input_token_details': {'cache_read': 0,
  'cache_creation': 0,
  'ephemeral_5m_input_tokens': 0,
  'ephemeral_1h_input_tokens': 0}}

In [5]:
#print out all messages

import json

def print_aimessage(ai_message):
    for key, value in ai_message.model_dump().items():                #convert Ai-Message to Python dictionary and get items
        if isinstance(value, (dict, list)):                         #check if the value is a nested structure
            print(f"\n[{key}]:")                                    
            print(json.dumps(value, indent=2, ensure_ascii=False))  #converts a Python object into a JSON-formatted string. It’s commonly used to serialize data for storage or sending
        else:
            print(f"\n[{key}]: {value}")

print_aimessage(response)


[content]: # Pizza Order Transcription

**Customer:** Hi, I'd like to place an order for delivery.

**Operator:** Sure! What can I get started for you today?

**Customer:** I'd like a large pepperoni pizza and a medium vegetarian with extra mushrooms, please.

**Operator:** Great! One large pepperoni and one medium veggie with extra mushrooms. Anything else?

**Customer:** Yeah, can we add a 2-liter bottle of Coke and an order of garlic bread?

**Operator:** Perfect. So that's a large pepperoni, medium vegetarian with extra mushrooms, one 2-liter Coke, and garlic bread. What's the delivery address?

**Customer:** 247 Maple Street, Apartment 3B.

**Operator:** 247 Maple Street, Apartment 3B. And what's

[additional_kwargs]:
{}

[response_metadata]:
{
  "id": "msg_011Cf3JyqVi3traeVpQL7Jvi",
  "container": null,
  "model": "claude-haiku-4-5-20251001",
  "stop_details": null,
  "stop_reason": "max_tokens",
  "stop_sequence": null,
  "usage": {
    "cache_creation": {
      "ephemeral_1h_i

In [7]:
#start the ollama server
#uv add langchain_ollama
# utliza un modelo descargado

model_name = "qwen3:4b-instruct"
provider = "ollama"
prompt = "Generate the transcription of a pizza order"

response = generate_example_orders(
    model_name,
    prompt,
    provider
)

print_aimessage(response)


[content]: Sure! Here's a natural and clear transcription of a pizza order, as if a customer is speaking to a restaurant or ordering over the phone:

---

**Customer:**  
Hi, I’d like to order a pizza, please.  
Can I get a large pepperoni pizza with extra cheese?  

**Waiter/Operator:**  
Sure, one large pepperoni pizza with extra cheese, okay?  

**Customer:**  
Yes, and could you also add a side of garlic bread?  

**Waiter/Operator:**  
Absolutely, one large pepperoni pizza with extra cheese and one side of garlic bread.  

**Customer:**  
Perfect. And could you please deliver it to 123 Main Street?  

**Waiter/Operator:**  
Of course. Delivery to 123 Main Street.  

**Customer:**  
Great. How long will it take to arrive?  

**Waiter/Operator:**  
It should arrive in about 30 to 40 minutes, depending on traffic.  

**Customer:**  
Alright, thanks! I’ll be waiting.  

**Waiter/Operator:**  
You're welcome! Your order is confirmed. We’ll have it ready and out for delivery shortly.  

In [ ]:
import time
import pandas as pd
from pathlib import Path
from langchain.chat_models import init_chat_model

# 1. Define model configurations
#    Lista ajustada a los modelos realmente descargados con `ollama pull`.
#    claude-haiku necesita ANTHROPIC_API_KEY en el .env; si no la tienes,
#    el try/except de abajo lo salta sin romper el benchmark.
models_config = [
    # Ollama Models (locales)
    {"model": "qwen3:0.6b", "provider": "ollama"},
    {"model": "qwen3:1.7b", "provider": "ollama"},
    {"model": "llama3.2:3b", "provider": "ollama"},
    {"model": "gemma3:4b", "provider": "ollama"},
    {"model": "mistral:7b", "provider": "ollama"},
    {"model": "qwen3:8b", "provider": "ollama"},

    # Anthropic Model (en la nube, requiere API key)
    {"model": "claude-haiku-4-5-20251001", "provider": "anthropic"},
]

prompt = "Generate the transcription of a pizza order."
results = []



# 2. Invoke models and record metadata
for config in models_config:
    model_name = config["model"]
    provider = config["provider"]
    temperature = 1.0
    
    try:
        # Initialize LLM via LangChain init_chat_model
        llm = init_chat_model(model_name, model_provider=provider, temperature=temperature)
        
        # Track wall-clock duration
        start_time = time.perf_counter()
        response = llm.invoke(prompt)
        end_time = time.perf_counter()
        
        duration = round(end_time - start_time, 3)
        answer = response.content
        
        # Extract usage metadata
        usage_metadata = getattr(response, "usage_metadata", {}) or {}
        input_tokens = usage_metadata.get("input_tokens", None)
        output_tokens = usage_metadata.get("output_tokens", None)
        
        
        results.append({
            "modelname": model_name,
            "provider": provider,
            "duration_sec": duration,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "answer": answer
        })
        print(f"Executed {model_name} in {duration} seconds. Input tokens: {input_tokens}, Output tokens: {output_tokens}, Answer: {answer[:80]}...")  # Print first 50 chars of answer for brevity
        
    except Exception as e:
        print(f"Error executing {model_name}: {e}")

# 3. Create Pandas DataFrame
df = pd.DataFrame(results)

df

In [13]:
# save df to csv
from datetime import datetime
from pathlib import Path

script_dir = Path.cwd()
data_path = script_dir.parent / "data" / "performance"

data_path.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = data_path / f"performance_test_{timestamp}.csv"
df.to_csv(data_path / output_file, index=False)

In [11]:
df.sort_values(by="duration_sec", ascending=True, inplace=False)

,modelname,provider,duration_sec,input_tokens,output_tokens,answer
10,claude-haiku-4-5-20251001,anthropic,2.992,16,250,# Pizza Order Transcription\n\n**Customer:** H...
7,qwen3:4b-instruct,ollama,34.381,16,327,Sure! Here's a natural and clear transcription...
3,ministral-3:3b,ollama,36.227,561,237,Here’s a professional and clear transcription ...
2,llama3.1:8b,ollama,43.928,18,190,Here is the transcription of a pizza order:\n\...
0,gemma3:4b,ollama,47.764,18,564,"Okay, here's a transcription of a pizza order,..."
4,mistral:7b,ollama,53.933,13,335,"Customer: Hi, I'd like to place an order plea..."
6,qwen3.5:0.8b,ollama,58.905,18,1607,Here is a structured transcription of a pizza ...
8,qwen3:8b,ollama,97.163,18,506,**Pizza Order Transcription** \n\n**Customer:...
5,mistral-nemo:12b,ollama,108.973,11,453,"**Order Taker (OT):** Hello, Pizza Palace! How..."
1,gemma4:e4b,ollama,155.351,24,1482,"Since a ""transcription"" can mean either the di..."


In [14]:
answer_list = df ["answer"].to_list()
for idx, answer in enumerate(answer_list):
    print(f"Answer from model {df['modelname'][idx]} Output tokens: {df['output_tokens'][idx]}:\n {answer}\n")
    print("------------------------------------------------------------\n")

Answer from model gemma3:4b Output tokens: 716:
 Okay, here’s a transcription of a typical pizza order, aiming for a natural-sounding conversation:

**Setting:** Phone call between a customer (C) and a Pizza Place employee (P)

**(Sound of a phone ringing, then being answered)**

**P:** Thank you for calling Tony’s Pizza, this is Mark speaking, how can I help you?

**C:** Hi Mark, yeah, I’d like to place an order for pickup.

**P:** Great! Can I get your name and phone number, please?

**C:** It’s Sarah Miller, and my number is 555-123-4567.

**P:** Okay, Sarah, what can I get for you today?

**C:** Hmm… I’ll take a large pepperoni pizza, please.

**P:** Okay, large pepperoni. Anything else on that?

**C:** Yeah, can I add some mushrooms?  And maybe a little extra cheese?

**P:** Sure thing. Large pepperoni with mushrooms and extra cheese.  Any sauce preference – regular or spicy?

**C:** Regular is fine.

**P:** Perfect. And what size for your second pizza?

**C:** Let’s go with a med

In [ ]:
#to manage results more easely we only want the turn with the order.
#adapt prompt: Only generate the customers turn with the order.
model_name = "qwen3:1.7b"
provider = "ollama"
prompt = """
Generate a pizza order transcribed as if the customer was placing them verbally over the phone. 
Only generate the customers turn with the order.
"""


# Ojo: el 4o parametro posicional de generate_example_orders es token_limit,
# no temperature. Pasamos temperature con nombre para evitar el bug.
response = generate_example_orders(
    model_name,
    prompt,
    provider,
    temperature=0.7
)

print(response.content)
print(len(response.content))


In [ ]:
# The standard way to ensure consistent output format from LLMs is to use Pydantic models to define and validate the expected structure, 
# which can then be enforced via the API's structured-output feature

from typing import List
from pydantic import BaseModel, Field

class Pizza(BaseModel):
    """Transcription of the pizza order."""

    pizza_order: str = Field(
        ..., description="The pizza order transcription."
    )

class Pizza_order_list(BaseModel):
    """ Generated Pizza orders"""
    # Creates a model so that we can extract multiple entities.
    order_list: List[Pizza]



In [ ]:
from langchain.chat_models import init_chat_model

def generate_structured_example_orders(model_name, prompt, provider=None, temperature = 0.7):
    model = init_chat_model(
        model_name,
        model_provider=provider,
        temperature = temperature
    )
    structured_llm = model.with_structured_output(schema=Pizza_order_list, include_raw=True)
    response = structured_llm.invoke(prompt)
    return response

In [ ]:
model_name = "qwen3:8b"
provider = "ollama"
prompt = """
Generate 10 pizza orders transcribed as if the customer was placing them verbally over the phone.
Only generate the customer's turn with the order.

Make the orders demanding and hard to parse. Each order must include at least two of:
- changing their mind mid-sentence and correcting themselves
- a half-and-half pizza with different toppings on each side
- removing a default ingredient ("no onions", "hold the cheese")
- an allergy or dietary restriction
- unusual quantities or sizes mixed in one order
- hesitations and filler words ("erm", "let me think", "actually...")

Vary the length and the customer's tone across the ten orders.
"""

response = generate_structured_example_orders(
    model_name,
    prompt,
    provider,
    temperature=0.9
)
response

In [ ]:
#transform pydantic object to Python dict using model_dump()
print(response["raw"].usage_metadata)
print(response["parsed"])
response_dict = response["parsed"].model_dump()
print(type(response_dict))
response_dict

In [ ]:
#get the values of the dict 
for order in response_dict['order_list']:
    print(order ['pizza_order'])

In [ ]:
# save  response (pydantic object) as json 
from datetime import datetime
from pathlib import Path


def save_orders_to_file(content) -> Path:
    data_path = Path.cwd().parent / "data" / "example_orders"
    data_path.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_file = data_path / f"pedidos_pizza_{timestamp}.json"

    output_file.write_text(
        content.model_dump_json(indent=2),
        encoding="utf-8",
    )

    return output_file

In [ ]:
save_orders_to_file(response)

In [ ]:
from pathlib import Path


def load_orders_as_list(filename: str) -> list[str]:
    """Lädt eine JSON-Datei aus data/example_orders und gibt die Orders als Liste zurück."""
    data_path = Path.cwd().parent / "data" / "example_orders"
    path = data_path / filename
    data = json.loads(path.read_text(encoding="utf-8"))

    return [order["pizza_order"] for order in data["order_list"]]

    


In [ ]:
# Cargar automaticamente el ultimo JSON de pedidos generado
# (evita el nombre fijo pedidos_pizza_20260914_115625.json que era del profesor)
from pathlib import Path
orders_dir = Path.cwd().parent / "data" / "example_orders"
latest = sorted(orders_dir.glob("pedidos_pizza_*.json"))[-1]
print("Cargando:", latest.name)

python_list = load_orders_as_list(latest.name)
print(type(python_list))
python_list
